# Pyspark Introduction

In [0]:
# Loading the olympics dataset

DATA_PATH = "/Volumes/data/olympic_games/raw_data"

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header = True)

df_athletes

In [0]:
df_athletes.limit(2).display()

In [0]:
df_athletes.printSchema()

In [0]:
#  Length of the dataframe

(df_athletes.count(), len(df_athletes.columns))


# Infer the Schema

NOTE: Age, Height = STRINGS type because it has NULL values

In [0]:
df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header = True)
df_athletes

In [0]:
display(df_athletes)

## Define Explicit Schema

In [0]:

from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", ShortType()),
  StructField("Weight", ShortType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])



df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)
     

## EDA

In [0]:
# PySpark Transformations

df_athletes_schema.groupBy("NOC", "Medal").count().filter("NOC IN ('SWE', 'NOR', 'FIN', 'ISL', 'DEN') AND Medal != 'NA'").sort("NOC", "Medal").display()

## Spark SQL Transformations


In [0]:
# Register the df as a SQL Object
df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

# Read with spark sql
spark.sql("FROM df_athletes_schema")

# Pickout Swedish Medals in Table Tennis

spark.sql("SELECT * FROM df_athletes_schema WHERE NOC = 'SWE' AND Sport = 'Table Tennis' and Medal != 'NA'").display()


In [0]:
# Count Medals and Plot

df_swe_medals = spark.sql("""
          SELECT 
            sport,
          count(*) as count 
          FROM df_athletes_schema
          WHERE NOC = 'SWE' AND Medal IN ('Bronze', 'Silver', 'Gold')
          GROUP BY sport
          ORDER BY count DESC
          """)

df_swe_medals.display()

In [0]:
df_swe_medals.plot(kind = "bar", y = "sport", x = "count")